In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import sys
import os

# Tell the notebook to look at the 'src' folder
sys.path.append('../')
from src.model import get_model

In [2]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),       # Standard size for MobileNet
    transforms.RandomHorizontalFlip(),   # Flips images for variety
    transforms.ToTensor(),               # Converts image to numbers
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]) # Standard AI scaling
])

# Load your clean data
dataset = datasets.ImageFolder('../data/processed', transform=transform)
train_loader = DataLoader(dataset, batch_size=32, shuffle=True)

In [4]:
# Check if Apple Silicon GPU (MPS) is available
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

# Move your model to the GPU
model = get_model(num_classes=2).to(device)

# Inside your loop, move images and labels to the GPU too
for images, labels in train_loader:
    images, labels = images.to(device), labels.to(device)
    # ... rest of your code ...

Using device: mps


### Training loop for the model

In [5]:
model = get_model(num_classes=2)
criterion = nn.CrossEntropyLoss() # The "Scolding" function (calculates error)
optimizer = optim.Adam(model.parameters(), lr=0.001) # The "Optimizer" (fixes the error)

epochs = 5 # Start with 5 rounds through the dataset

for epoch in range(epochs):
    running_loss = 0.0
    for images, labels in train_loader:
        optimizer.zero_grad()           # Reset the math
        outputs = model(images)         # Make a guess
        loss = criterion(outputs, labels) # How wrong was the guess?
        loss.backward()                 # Calculate how to fix it
        optimizer.step()                # Apply the fix
        running_loss += loss.item()
    
    print(f"Epoch {epoch+1}/{epochs} - Loss: {running_loss/len(train_loader):.4f}")

Epoch 1/5 - Loss: 0.0634
Epoch 2/5 - Loss: 0.0238
Epoch 3/5 - Loss: 0.0123
Epoch 4/5 - Loss: 0.0140
Epoch 5/5 - Loss: 0.0117


### Saving the model

In [6]:
os.makedirs("../models", exist_ok=True)
torch.save(model.state_dict(), "../models/best_model.pth")
print("Model saved to models/best_model.pth!")

Model saved to models/best_model.pth!
